In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ['GROQ_API_KEY']:
    print("API KEY SET")

API KEY SET


In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

llm.invoke("Hello!").content

f:\AI_Projects\rag_basics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'Hello. How can I assist you today?'

## **RAG IMPLEMENTATION WITH PDF's** 

## STEP 1 - Extracting Text From PDF's

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./Docs/Module 2.pdf"

loader = PyPDFLoader(pdf_path)

docs = loader.load()

docs

Overwriting cache for 0 310


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2020-06-05T13:01:54-07:00', 'moddate': '2020-11-25T00:01:48+05:30', 'source': './Docs/Module 2.pdf', 'total_pages': 106, 'page': 0, 'page_label': '1'}, page_content='VTUPulse.com \n \n \n \nMANAGEMENT & ENTREPRENEURSHIP                                                                            MODULE 2 \n \nCBIT Kolar Page 1 \n \n \n \n \n \nTechnological  Innovation  \nManagement & Entrepreneurship  \nB.E., V Semester, Electronics & Communication Engineering \n[As per Choice Based Credit System (CBCS) scheme]  \n[Subject code: 18ES51] \n     \n  \n \n \n \n \n \nMODULE - 2a \n \n \n \nORGANISING AND STAFFING'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2020-06-05T13:01:54-07:00', 'moddate': '2020-11-25T00:01:48+05:30', 'source': './Docs/Module 2.pdf', 'total_pages': 106, 'page': 1, 'page_label': '2'}, page_content='VTUPulse.com \n \n \n \nMANAGEMENT & ENTREPRENEURSHIP     

## STEP 2 - Splitting the Document into chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)

chunks = splitter.split_documents(docs)

chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2020-06-05T13:01:54-07:00', 'moddate': '2020-11-25T00:01:48+05:30', 'source': './Docs/Module 2.pdf', 'total_pages': 106, 'page': 0, 'page_label': '1'}, page_content='VTUPulse.com \n \n \n \nMANAGEMENT & ENTREPRENEURSHIP                                                                            MODULE 2 \n \nCBIT Kolar Page 1 \n \n \n \n \n \nTechnological  Innovation  \nManagement & Entrepreneurship  \nB.E., V Semester, Electronics & Communication Engineering \n[As per Choice Based Credit System (CBCS) scheme]  \n[Subject code: 18ES51] \n     \n  \n \n \n \n \n \nMODULE - 2a \n \n \n \nORGANISING AND STAFFING'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2020-06-05T13:01:54-07:00', 'moddate': '2020-11-25T00:01:48+05:30', 'source': './Docs/Module 2.pdf', 'total_pages': 106, 'page': 1, 'page_label': '2'}, page_content='VTUPulse.com \n \n \n \nMANAGEMENT & ENTREPRENEURSHIP     

In [5]:
len(chunks)

201

In [6]:
#PyPDF Loader Will Also Create Metadata

chunks[0].metadata

{'producer': 'PyPDF',
 'creator': 'PyPDF',
 'creationdate': '2020-06-05T13:01:54-07:00',
 'moddate': '2020-11-25T00:01:48+05:30',
 'source': './Docs/Module 2.pdf',
 'total_pages': 106,
 'page': 0,
 'page_label': '1'}

## **Creating Own MetaData For PDF Chunks**

In [12]:
for i in docs:

    i.metadata = {"source" : "Module 1.pdf",
                  "developer" : "Microsoft"}

In [13]:
docs

[Document(metadata={'source': 'Module 1.pdf', 'developer': 'Microsoft'}, page_content='VTUPulse.com \n \n \n \nMANAGEMENT & ENTREPRENEURSHIP MODULE 1\nCBIT-KOLAR Page 1\nTechnological Innovation\nManagement & Entrepreneurship\nB.E., V Semester, Electronics & Communication\nEngineering\n[As per Choice Based Credit System (CBCS) scheme]\n[Subject Code: 18ES51]\nMODULE – 1a\nMANAGEMENT\nDefinition of management:\n \n \n \n \n \nVTUPulse.com'),
 Document(metadata={'source': 'Module 1.pdf', 'developer': 'Microsoft'}, page_content='VTUPulse.com \n \n \n \nMANAGEMENT & ENTREPRENEURSHIP MODULE 1\nCBIT-KOLAR Page 2\nManagement is a function of guidance and leadership control of efforts of a\ngroup or individuals in order to achieve goals/objectives of an organization.\nSimplest definition is that it is defined as the art of getting things done\nthrough people. Management can also be defined as The process consisting of\nplanning, organizing, actuating, and controlling performed to determine and

## STEP 3 - Creating Embeddings for the chunks

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 22689.77it/s]


## STEP 4 - Create & Store Embeddings In The Existing Local Vector Store

In [8]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(
    persist_directory="./Vector/",
    embedding_function=embedding_model
)

C:\Users\harsh\AppData\Local\Temp\ipykernel_1868\1799798528.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [9]:
vectorstore.add_documents(chunks)

['f1983d72-415f-4ba6-8674-d816fd97c261',
 '2b9e4925-5fee-436a-b43a-1146004f7850',
 '2d0d3d9d-c658-49ff-b63e-5f2db3d01a2a',
 '29fe5b79-ebdd-42a0-8206-b041a5a64d8b',
 'a7cdfdbc-c43c-42c8-8306-3c49270edc50',
 '3d021f4c-92ec-4bba-8fe8-7b20c0a0be13',
 'aed5c833-354e-42f2-b07a-8ccbefa6a172',
 '625802d8-d5f7-4025-a3cd-f8c13ab8a37c',
 '36bfb435-944b-4f34-b427-9cf6e21ee28f',
 '9b63f7f6-1b2b-4f84-94ec-57a57965c027',
 'd7a06c54-4e9e-4e19-9d6c-aeb5723edce8',
 '2b85abaa-f128-4e93-82b3-de54a2660293',
 '77a6368f-6dc4-454a-acee-768835160f5a',
 'bf201f17-9856-4d7f-803b-f1b762e83776',
 'f3a59afa-de7c-48c6-abce-ac4d2cea7578',
 '44285124-2b5b-4b65-8928-6d732ea26545',
 'a3a8e0ee-b074-49dc-a13d-9abcbabeeb57',
 '6ddf9d9b-0025-4367-826c-f6c0fc82c227',
 'c92c75c1-b3cb-4e18-bbd1-521b84c60921',
 'da865193-57e7-4534-b1cd-f86f9abb542a',
 'ca7be2d1-e18a-46ac-9c70-9b02fd1383a6',
 'b7b9c3ee-dfb9-443f-ae00-ce289fe5cb29',
 '3197122a-892e-476a-88a1-623a94b6281c',
 '88595a8c-7ca5-49a4-8e2e-09b7359c07e0',
 '23aa1047-6ca6-

## STEP 5 - Semantic Search

In [10]:
vectorstore.similarity_search("What are the principles of organization?", k=5)

[Document(metadata={'total_pages': 106, 'page': 8, 'page_label': '9', 'moddate': '2020-11-25T00:01:48+05:30', 'creationdate': '2020-06-05T13:01:54-07:00', 'source': './Docs/Module 2.pdf', 'creator': 'PyPDF', 'producer': 'PyPDF'}, page_content='VTUPulse.com \n \n \n \nMANAGEMENT & ENTREPRENEURSHIP                                                                            MODULE 2 \n \nCBIT Kolar Page 9 \n \nPRINCIPLES OF ORGANIZATION \nThe success of a business organization can be \nensured if the following basic principles are used. In order \nto develop a sound and efficient organization structure, \nthere is need to follow certain principles. In the words of \nE.F.L. Brech, "If there is to be a systematic approach to \nthe formulation of organization structure, there ought to be \na body of accepted principles". They are as follows: \n(1) Objectives: The objectives of the enterprise influence \nthe organization structure and hence the objectives of \nthe enterprise should first be cl

In [11]:
context = vectorstore.similarity_search("What are the principles of organization?", k=5)

response = llm.invoke(f"What are the principles of organization? You can answer using the following context: {context}")

print(response.content)

Based on the provided context, the principles of organization are as follows:

1. **Objectives**: The objectives of the enterprise influence the organization structure, and hence, the objectives of the enterprise should first be clearly defined. Every part of the organization should be geared to the achievement of these objectives.

2. **Specialisation**: Effective organization must promote specialization. The activities of the enterprise should be coordinated at various levels.

3. **Coordination**: Coordination of activities at various levels is essential for effective organization.

4. **Personal Ability**: As people constitute an organization, there is a need for proper selection, placement, and training of staff. Organization structure must ensure optimum use of human resources and encourage management development programs.

5. **Organization as an Open System**: An organization is an open system, without which it cannot survive. It interacts with its environment to achieve its go